# 🚁 DIP Project — Gün 12: SAHI'lı vs SAHI'sız Görsel Karşılaştırma
Aynı VisDrone görüntüsü üzerinde iki yaklaşımı yan yana karşılaştır.
Küçük nesne tespitindeki farkı görselleştir.

> **Gereksinim:** Gün 5 modeli hazır olmalı.

In [3]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(args))

pip('--force-reinstall', 'opencv-python-headless==4.10.0.84')
pip('ultralytics>=8.3.0', 'sahi==0.11.15')

print('✅ Kurulum tamam — Runtime > Restart Runtime yap')


✅ Kurulum tamam — Runtime > Restart Runtime yap


In [1]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(args))

# Önce eski cv2'yi cache'den temizle
for key in list(sys.modules.keys()):
    if 'cv2' in key:
        del sys.modules[key]

pip('--force-reinstall', 'opencv-python-headless==4.10.0.84')
pip('ultralytics>=8.3.0', 'sahi==0.11.15')

import importlib
importlib.invalidate_caches()

# Hemen aynı hücrede import et
import cv2
import numpy as np
print(f'✅ numpy {np.__version__}')
print(f'✅ cv2   {cv2.__version__}')
print('✅ Sonraki hücreye geç')


✅ numpy 2.4.6
✅ cv2   4.10.0
✅ Sonraki hücreye geç


In [8]:
MODEL_PATH   = '/content/drive/MyDrive/DIP_Project/runs/yolov8s_visdrone_v1/weights/best.pt'
PROJECT_DIR  = '/content/drive/MyDrive/DIP_Project'
DATASET_ROOT = f'{PROJECT_DIR}/datasets/VisDrone'
YAML_PATH   = f'{PROJECT_DIR}/data.yaml'
import os
print('MODEL_PATH:', MODEL_PATH)
print('Dosya var mı:', os.path.exists(MODEL_PATH))


MODEL_PATH: /content/drive/MyDrive/DIP_Project/runs/yolov8s_visdrone_v1/weights/best.pt
Dosya var mı: True


In [4]:
import sahi
print(sahi.__version__)

# Desteklenen model tipleri
from sahi.auto_model import MODEL_TYPE_TO_MODEL_CLASS_NAME
print(list(MODEL_TYPE_TO_MODEL_CLASS_NAME.keys()))


0.11.15
['yolov8', 'mmdet', 'yolov5', 'detectron2', 'huggingface', 'torchvision', 'yolov5sparse', 'yolonas']


In [6]:
PROJECT_DIR  = '/content/drive/MyDrive/DIP_Project'
DATASET_ROOT = f'{PROJECT_DIR}/datasets/VisDrone'
MODEL_PATH   = f'{PROJECT_DIR}/runs/yolov8s_visdrone_v1/weights/best.pt'


In [10]:
# ▶️ Restart sonrası buradan başla
import torch, os, cv2, json, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction, get_prediction

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Device: {DEVICE}')

from google.colab import drive
drive.mount('/content/drive')

meta_path = f'{PROJECT_DIR}/model_paths.json'
if os.path.exists(meta_path):
    with open(meta_path) as f:
        MODEL_PATH = json.load(f)['best_model']
else:
    MODEL_PATH = 'yolov8n.pt'
    print('⚠️  Gün 5 modeli bulunamadı — yolov8n.pt ile devam')

sahi_model = AutoDetectionModel.from_pretrained(
    model_type='yolov8',
    model_path='/content/drive/MyDrive/DIP_Project/runs/yolov8s_visdrone_v1/weights/best.pt',
    confidence_threshold=0.25,
    device=DEVICE,
)
yolo_model = YOLO('/content/drive/MyDrive/DIP_Project/runs/yolov8s_visdrone_v1/weights/best.pt')
print('✅ Modeller yüklendi')




✅ Device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Modeller yüklendi


## 1️⃣ Test Görüntülerini Seç — Küçük Nesne Yoğun Sahneler

In [11]:
val_dir  = Path(f'{DATASET_ROOT}/images/val')
lbl_dir  = Path(f'{DATASET_ROOT}/labels/val')

test_imgs = []

if val_dir.exists() and lbl_dir.exists():
    # Küçük nesne sayısı en fazla olan görüntüleri seç
    candidates = []
    for lf in sorted(lbl_dir.glob('*.txt'))[:200]:
        ip = val_dir / (lf.stem + '.jpg')
        if not ip.exists(): continue
        img = cv2.imread(str(ip))
        if img is None: continue
        ih, iw = img.shape[:2]
        small_count = 0
        with open(lf) as f:
            for line in f:
                p = line.strip().split()
                if len(p) < 5: continue
                w = float(p[3]) * iw
                h = float(p[4]) * ih
                if w < 32 and h < 32:
                    small_count += 1
        candidates.append((small_count, str(ip)))
    candidates.sort(key=lambda x: -x[0])
    test_imgs = [c[1] for c in candidates[:5]]
    print(f'✅ Küçük nesne yoğun {len(test_imgs)} görüntü seçildi')
    for cnt, path in candidates[:5]:
        print(f'   {Path(path).name}: {cnt} küçük nesne')

if not test_imgs:
    # Fallback: ilk 5 val görüntüsü
    if val_dir.exists():
        test_imgs = [str(f) for f in sorted(val_dir.glob('*.jpg'))[:5]]
    else:
        import urllib.request
        urllib.request.urlretrieve('https://ultralytics.com/images/zidane.jpg', '/content/test.jpg')
        test_imgs = ['/content/test.jpg']
    print(f'⚠️  Fallback: {len(test_imgs)} görüntü')

✅ Küçük nesne yoğun 5 görüntü seçildi
   0000086_01954_d_0000005.jpg: 139 küçük nesne
   0000194_00399_d_0000121.jpg: 132 küçük nesne
   0000215_00000_d_0000256.jpg: 123 küçük nesne
   0000244_02000_d_0000005.jpg: 117 küçük nesne
   0000244_03500_d_0000008.jpg: 113 küçük nesne


## 2️⃣ Karşılaştırmalı Inference — SAHI'sız vs SAHI'lı

In [12]:
COLORS = {
    'no_sahi': (255,  80,  80),   # kırmızı
    'sahi':    ( 50, 200, 100),   # yeşil
}

comparison_data = []

for img_path in test_imgs:
    frame = cv2.imread(img_path)
    if frame is None: continue

    # --- SAHI'sız ---
    t0     = time.time()
    result = yolo_model.predict(frame, conf=0.25, device=DEVICE, verbose=False)[0]
    t_nosahi = time.time() - t0
    n_nosahi = len(result.boxes)

    frame_nosahi = frame.copy()
    for box in result.boxes.xyxy.cpu().numpy():
        x1,y1,x2,y2 = map(int, box[:4])
        cv2.rectangle(frame_nosahi, (x1,y1), (x2,y2), COLORS['no_sahi'], 2)

    # --- SAHI'lı ---
    t1 = time.time()
    sahi_result = get_sliced_prediction(
        img_path, sahi_model,
        slice_height=640, slice_width=640,
        overlap_height_ratio=0.2, overlap_width_ratio=0.2,
        verbose=0,
    )
    t_sahi = time.time() - t1
    n_sahi = len(sahi_result.object_prediction_list)

    frame_sahi = frame.copy()
    for pred in sahi_result.object_prediction_list:
        bbox = pred.bbox
        x1,y1 = int(bbox.minx), int(bbox.miny)
        x2,y2 = int(bbox.maxx), int(bbox.maxy)
        cv2.rectangle(frame_sahi, (x1,y1), (x2,y2), COLORS['sahi'], 2)

    comparison_data.append({
        'img_name':   Path(img_path).name,
        'n_nosahi':   n_nosahi,
        'n_sahi':     n_sahi,
        'delta':      n_sahi - n_nosahi,
        't_nosahi_ms': t_nosahi * 1000,
        't_sahi_ms':   t_sahi   * 1000,
        'frame_nosahi': frame_nosahi,
        'frame_sahi':   frame_sahi,
        'orig_img':     frame,
    })
    print(f'  {Path(img_path).name}: '
          f'No-SAHI={n_nosahi} ({t_nosahi*1000:.0f}ms) | '
          f'SAHI={n_sahi} ({t_sahi*1000:.0f}ms) | '
          f'Δ=+{n_sahi-n_nosahi}')

print('\n✅ Karşılaştırma tamamlandı!')

  0000086_01954_d_0000005.jpg: No-SAHI=106 (1409ms) | SAHI=115 (338ms) | Δ=+9
  0000194_00399_d_0000121.jpg: No-SAHI=94 (14ms) | SAHI=108 (136ms) | Δ=+14
  0000215_00000_d_0000256.jpg: No-SAHI=104 (14ms) | SAHI=159 (366ms) | Δ=+55
  0000244_02000_d_0000005.jpg: No-SAHI=82 (19ms) | SAHI=105 (161ms) | Δ=+23
  0000244_03500_d_0000008.jpg: No-SAHI=56 (16ms) | SAHI=85 (147ms) | Δ=+29

✅ Karşılaştırma tamamlandı!


## 3️⃣ Side-by-Side Görsel Karşılaştırma

In [13]:
OUT_DIR = f'{PROJECT_DIR}/sahi_comparison'
os.makedirs(OUT_DIR, exist_ok=True)

for i, d in enumerate(comparison_data):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(cv2.cvtColor(d['orig_img'],     cv2.COLOR_BGR2RGB))
    axes[0].set_title('Orijinal Görüntü', fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(cv2.cvtColor(d['frame_nosahi'], cv2.COLOR_BGR2RGB))
    axes[1].set_title(
        f'YOLOv8 (SAHI sız)\n{d["n_nosahi"]} tespit | {d["t_nosahi_ms"]:.0f}ms',
        fontweight='bold', color='#c0392b'
    )
    axes[1].axis('off')

    axes[2].imshow(cv2.cvtColor(d['frame_sahi'],   cv2.COLOR_BGR2RGB))
    axes[2].set_title(
        f'YOLOv8 + SAHI (640x640 %20)\n{d["n_sahi"]} tespit | {d["t_sahi_ms"]:.0f}ms',
        fontweight='bold', color='#27ae60'
    )
    axes[2].axis('off')

    plt.suptitle(
        f'{d["img_name"]} — Δ = +{d["delta"]} ek tespit (SAHI ile)',
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    out_path = f'{OUT_DIR}/compare_{i+1:02d}_{d["img_name"]}.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Kaydedildi: {out_path}')

Output hidden; open in https://colab.research.google.com to view.

## 4️⃣ Özet İstatistik Tablosu

In [14]:
import pandas as pd

summary_rows = []
for d in comparison_data:
    summary_rows.append({
        'Görüntü':           d['img_name'],
        'SAHI sız Tespit':   d['n_nosahi'],
        'SAHI lı Tespit':    d['n_sahi'],
        'Fark (+)':          d['delta'],
        'SAHI sız (ms)':     round(d['t_nosahi_ms'], 1),
        'SAHI lı (ms)':      round(d['t_sahi_ms'],   1),
        'Hız Farkı (x)':     round(d['t_sahi_ms'] / max(d['t_nosahi_ms'], 1), 1),
    })

df = pd.DataFrame(summary_rows)
print('\n📊 SAHI Karşılaştırma Özeti:')
print(df.to_string(index=False))

print(f'\n  Ortalama ek tespit (SAHI ile): +{df["Fark (+)"].mean():.1f}')
print(f'  Ortalama hız farkı            : {df["Hız Farkı (x)"].mean():.1f}x yavaş')

df.to_csv(f'{PROJECT_DIR}/sahi_comparison_summary.csv', index=False)

print('\n✅ GÜN 12 TAMAMLANDI!')


📊 SAHI Karşılaştırma Özeti:
                    Görüntü  SAHI sız Tespit  SAHI lı Tespit  Fark (+)  SAHI sız (ms)  SAHI lı (ms)  Hız Farkı (x)
0000086_01954_d_0000005.jpg              106             115         9         1409.3         337.7            0.2
0000194_00399_d_0000121.jpg               94             108        14           14.1         136.4            9.6
0000215_00000_d_0000256.jpg              104             159        55           14.3         366.1           25.6
0000244_02000_d_0000005.jpg               82             105        23           19.5         161.1            8.3
0000244_03500_d_0000008.jpg               56              85        29           15.5         146.6            9.4

  Ortalama ek tespit (SAHI ile): +26.0
  Ortalama hız farkı            : 10.6x yavaş

✅ GÜN 12 TAMAMLANDI!


---
## ✅ Gün 12 Özet
| Karşılaştırma | SAHI sız | SAHI'lı (640×640 %20) |
|---------------|----------|----------------------|
| Tespit sayısı | Az | Daha fazla |
| Küçük nesne | Kaçırılan | Yakalanan |
| Hız | Hızlı | Yavaş (patch sayısına bağlı) |

**Sonraki (Gün 13):** GPU VRAM + GFLOPs profiling → `day13_profiling.ipynb`